# Category Alignment: LOC, Scopus, DOAJ, and SCImago

> This study was conducted within the framework of the **BLOOM project**.

## Overview

This notebook aligns subject classification vocabularies from four sources:

| Source | Format | Role |
|--------|--------|------|
| **Library of Congress (LOC)** | RDF/XML alignment files | Anchor vocabulary |
| **Scopus** | Turtle (`.ttl`) SKOS vocabulary | Target of LOC alignment |
| **SCImago** | CSV (`scimagojr`) | Cross-check of Scopus areas/categories |
| **DOAJ** | CSV | Cross-check of subject string formatting |

The main output is a JSON file (`merged_loc_scopus.json`) in which every LOC entry is enriched with the Scopus categories and subject areas it aligns to, together with the alignment type (`closeMatch`, `narrowMatch`, `broadMatch`, or `relatedMatch`).

---

## Repository structure

Before running this notebook, make sure your working directory looks like this:

```
project_root/
├── journal_data/
│   ├── doaj.csv              # DOAJ journal export (UTF-8, comma-separated)
│   ├── scimago.csv                  # SCImago journal rankings (semicolon-separated)
│   ├── vocabolario_SCOPUS_validated.ttl  # Scopus SKOS vocabulary (Turtle)
│   └── File LoC aggiornati con allineamento/               # Folder with LOC RDF/XML alignment files
│       ├── *.rdf
│       └── ...
└── cat_alignment.ipynb                # This notebook
```

> **Note:** All file paths in the configuration cell below are relative to the notebook's location.  
> You only need to edit **Section 4 – Configuration** to match your actual filenames.

---

## Requirements

Install the required packages with:

```bash
pip install rdflib
```

The following libraries are used (all others are part of the Python standard library):

| Package | Purpose |
|---------|---------|
| `rdflib` | Parse Turtle and RDF/XML files, navigate SKOS graphs |
| `csv` | Read DOAJ and SCImago CSV files |
| `re` | Strip quartile annotations (Q1–Q4) from SCImago categories |
| `json` | Serialise the merged output |
| `urllib.parse` | Percent-encode URIs that contain special characters |
| `os` | Iterate over the LOC alignment folder |
| `collections.defaultdict` | Build the in-memory Scopus graph |


## 1. Imports

In [1]:
import csv
import re
import json
import os
from collections import defaultdict
from urllib.parse import quote

from rdflib import Graph, Namespace, URIRef


## 2. Configuration

Edit the paths below to match your local setup.  
All paths are **relative to the notebook's location** so the notebook works on any machine.


In [2]:
# ── File paths (relative to this notebook) ────────────────────────────────────

DOAJ_CSV       = os.path.join("data", "doaj.csv")
SCIMAGO_CSV    = os.path.join("data", "scimago.csv")
SCOPUS_TTL     = os.path.join("data", "vocabolario_SCOPUS_validated.ttl")
LOC_FOLDER     = os.path.join("data", "File LoC aggiornati con allineamento")

OUTPUT_JSON    = "merged_loc_scopus.json"

# ── Sanity check ──────────────────────────────────────────────────────────────
for label, path in [
    ("DOAJ CSV",       DOAJ_CSV),
    ("SCImago CSV",    SCIMAGO_CSV),
    ("Scopus TTL",     SCOPUS_TTL),
    ("LOC folder",     LOC_FOLDER),
]:
    status = "✓ found" if os.path.exists(path) else "✗ NOT FOUND"
    print(f"{status}  {label}: {path}")


✓ found  DOAJ CSV: data\doaj.csv
✓ found  SCImago CSV: data\scimago.csv
✓ found  Scopus TTL: data\vocabolario_SCOPUS_validated.ttl
✓ found  LOC folder: data\File LoC aggiornati con allineamento


## 3. SCImago — Explore Areas and Categories

SCImago uses two levels of subject classification:

- **Areas** — broad subject groups (correspond to Scopus *subject areas*).
- **Categories** — narrower subjects within each area (correspond to Scopus *categories*).

Each row in the SCImago CSV can list multiple areas and categories, separated by semicolons.  
Categories also carry a quartile annotation (`Q1`–`Q4`) that we strip before comparison.

> **Finding:** The SCImago areas and categories align well with the Scopus SKOS vocabulary,
> but the ordering of the two columns is not always consistent — i.e. the first category
> in a row does not always belong to the first listed area.
> The `zip`-based pairing below is therefore a best-effort heuristic for rows with multiple areas.


In [3]:
def clean_category(cat: str) -> str:
    """Remove SCImago quartile annotations (e.g. '(Q1)') and strip whitespace."""
    return re.sub(r"\s*\(Q\d\)", "", cat).strip()


def map_areas_categories(input_csv: str, preview: int = 15) -> None:
    """
    Print a deduplicated, alphabetically sorted list of (Area, Category) pairs
    extracted from the SCImago CSV file.

    Parameters
    ----------
    input_csv : str
        Path to the SCImago CSV file (semicolon-separated).
    preview : int
        Number of rows to print (default: 15). Set to None to print all.
    """
    rows: set[tuple[str, str]] = set()

    with open(input_csv, encoding="utf-8") as infile:
        reader = csv.DictReader(infile, delimiter=";")

        for row in reader:
            areas      = [a.strip() for a in row.get("Areas", "").split(";") if a.strip()]
            categories = [clean_category(c) for c in row.get("Categories", "").split(";") if c.strip()]

            if not areas or not categories:
                continue

            if len(areas) == 1:
                # All categories belong to the single listed area
                for c in categories:
                    rows.add((areas[0], c))
            else:
                # Pair areas and categories positionally (best-effort)
                for area, category in zip(areas, categories):
                    rows.add((area, category))

    sorted_rows = sorted(rows, key=lambda x: x[0])

    for i, (area, category) in enumerate(sorted_rows):
        if preview is not None and i >= preview:
            print(f"  ... ({len(sorted_rows) - preview} more rows)")
            break
        print(f"{area};{category}")


map_areas_categories(SCIMAGO_CSV)


Agricultural and Biological Sciences;Plant Science
Agricultural and Biological Sciences;Arts and Humanities (miscellaneous)
Agricultural and Biological Sciences;Economics and Econometrics
Agricultural and Biological Sciences;Chemical Engineering (miscellaneous)
Agricultural and Biological Sciences;Conservation
Agricultural and Biological Sciences;Biomaterials
Agricultural and Biological Sciences;Ecology, Evolution, Behavior and Systematics
Agricultural and Biological Sciences;Chemical Health and Safety
Agricultural and Biological Sciences;Pharmacology, Toxicology and Pharmaceutics (miscellaneous)
Agricultural and Biological Sciences;Law
Agricultural and Biological Sciences;Computers in Earth Sciences
Agricultural and Biological Sciences;Soil Science
Agricultural and Biological Sciences;Visual Arts and Performing Arts
Agricultural and Biological Sciences;Nature and Landscape Conservation
Agricultural and Biological Sciences;Pharmaceutical Science
  ... (2779 more rows)


## 4. DOAJ — Inspect Subject String Format

Before mapping DOAJ journals to Scopus categories, we need to understand how DOAJ
encodes subject strings in its `Subjects` column.

> **Finding:** LOC uses `--` as the separator between a top-level class and its subclass  
> (e.g. `Philosophy. Psychology. Religion--Philosophy (General)`),  
> while DOAJ uses `:` for the same purpose  
> (e.g. `Philosophy. Psychology. Religion: Philosophy (General)`).  
> This difference must be normalised during any string-matching step.


In [4]:
def preview_doaj_subjects(doaj_csv: str, n: int = 15) -> None:
    """
    Print the first *n* non-empty Subjects values from the DOAJ CSV.

    Parameters
    ----------
    doaj_csv : str
        Path to the DOAJ CSV file (comma-separated, UTF-8).
    n : int
        Number of rows to preview (default: 15).
    """
    with open(doaj_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        count = 0
        for row in reader:
            subject = row.get("Subjects", "").strip()
            if subject:
                print(subject)
                count += 1
            if count >= n:
                break


preview_doaj_subjects(DOAJ_CSV)


Language and Literature: Greek language and literature. Latin language and literature
Law: Law of nations | Law: Law in general. Comparative and uniform law. Jurisprudence
Language and Literature: English language | Language and Literature: English literature
Language and Literature: Philology. Linguistics
Philosophy. Psychology. Religion: Philosophy (General)
Language and Literature: Philology. Linguistics
Medicine: Medicine (General)
Social Sciences: Commerce: Business: Accounting. Bookkeeping
Technology: Technology (General): Industrial engineering. Management engineering: Information technology
Geography. Anthropology. Recreation: Human ecology. Anthropogeography
Agriculture: Agriculture (General)
Law: Islamic law
Social Sciences
Philosophy. Psychology. Religion: Islam. Bahai Faith. Theosophy, etc.: Islam
Language and Literature: Philology. Linguistics


## 5. LOC–Scopus Alignment

This section contains the core pipeline:

1. **Parse the Scopus SKOS vocabulary** (`.ttl`) into an in-memory graph keyed by URI.
2. **Parse every LOC RDF/XML alignment file** in the configured folder.
3. **Merge** the two datasets: each LOC entry is enriched with the full Scopus metadata
   for every alignment URI it contains.
4. **Save** the result as a JSON file.

### 5.1 Namespace and utility helpers


In [5]:
# ── Namespaces ─────────────────────────────────────────────────────────────────
SKOS   = Namespace("http://www.w3.org/2004/02/skos/core#")
SKOSXL = Namespace("http://www.w3.org/2008/05/skos-xl#")

SCOPUS_NS = "https://www.scopus.com/vocabs/"
LOC_NS    = "http://id.loc.gov/authorities/classification/"


# ── Helpers ────────────────────────────────────────────────────────────────────

def is_scopus(uri: URIRef) -> bool:
    """Return True if the URI belongs to the Scopus vocabulary namespace."""
    return str(uri).startswith(SCOPUS_NS)


def is_loc(uri: URIRef) -> bool:
    """Return True if the URI belongs to the LOC classification namespace."""
    return str(uri).startswith(LOC_NS)


def create_safe_uri(uri: str) -> URIRef:
    """
    Percent-encode a URI while preserving common safe characters.
    Also removes escaped parentheses that can appear in Scopus URIs.
    """
    cleaned = uri.replace("\\(", "(").replace("\\)", ")")
    encoded = quote(cleaned, safe=":/#_-")
    return URIRef(encoded)


def clean_rdf_text(text: str) -> str:
    """Remove escaped parentheses that break RDF/XML parsing in some LOC files."""
    return text.replace("\\(", "(").replace("\\)", ")")


### 5.2 Parse the Scopus SKOS vocabulary

In [6]:
def parse_scopus_ttl(ttl_file: str) -> dict:
    """
    Parse the Scopus SKOS Turtle file and return a dictionary keyed by concept URI.

    Each entry contains:
    - ``label``           : preferred label of the concept
    - ``area``            : list of broader concept URIs (subject areas)
    - ``area_labels``     : preferred labels of those broader concepts
    - ``categories``      : list of narrower concept URIs
    - ``category_labels`` : preferred labels of those narrower concepts

    Parameters
    ----------
    ttl_file : str
        Path to the Scopus Turtle file.

    Returns
    -------
    dict[str, dict]
        Mapping from concept URI (str) to concept metadata.
    """
    print(f"Parsing Scopus TTL: {ttl_file}")

    g = Graph()
    g.parse(ttl_file, format="turtle")

    graph = defaultdict(lambda: {
        "label": None,
        "area": set(),
        "categories": set(),
        "area_labels": set(),
        "category_labels": set(),
    })

    for s in g.subjects():
        if not is_scopus(s):
            continue

        s_str = str(s)

        label = g.value(s, SKOS.prefLabel)
        if label:
            graph[s_str]["label"] = str(label)

        # skos:broader → subject area
        for o in g.objects(s, SKOS.broader):
            if is_scopus(o):
                graph[s_str]["area"].add(str(o))
                o_label = g.value(o, SKOS.prefLabel)
                if o_label:
                    graph[s_str]["area_labels"].add(str(o_label))

        # inverse skos:broader → narrower categories
        for o in g.subjects(SKOS.broader, s):
            if is_scopus(o):
                graph[s_str]["categories"].add(str(o))
                o_label = g.value(o, SKOS.prefLabel)
                if o_label:
                    graph[s_str]["category_labels"].add(str(o_label))

    # Convert sets to lists for JSON serialisability
    return {
        uri: {
            "label":           node["label"],
            "area":            list(node["area"]),
            "categories":      list(node["categories"]),
            "area_labels":     list(node["area_labels"]),
            "category_labels": list(node["category_labels"]),
        }
        for uri, node in graph.items()
    }


### 5.3 Parse LOC alignment files

In [7]:
def parse_rdf_file(file_path: str) -> list[dict]:
    """
    Parse a single LOC RDF/XML alignment file.

    Returns a list of LOC concept records, each containing:
    - ``loc_id``            : LOC concept URI
    - ``label``             : preferred label
    - ``alt_labels``        : alternative labels (SKOS and SKOS-XL)
    - ``scopus_alignments`` : list of dicts with ``uri`` and ``type`` (match type)

    Corrupted or unparseable files are skipped with a warning.
    """
    g = Graph()

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            rdf_text = f.read()

        rdf_text = clean_rdf_text(rdf_text)
        g.parse(data=rdf_text, format="xml")

    except Exception as e:
        print(f"  ⚠ Skipped (parse error): {os.path.basename(file_path)} — {e}")
        return []

    results = []

    for s in set(g.subjects()):
        if not is_loc(s):
            continue

        node = {
            "loc_id":            str(s),
            "label":             None,
            "alt_labels":        [],
            "scopus_alignments": [],
        }

        # Preferred label
        label = g.value(s, SKOS.prefLabel)
        if label:
            node["label"] = str(label)

        # Alternative labels — SKOS-XL
        for alt in g.objects(s, SKOSXL.altLabel):
            literal = g.value(alt, SKOSXL.literalForm)
            if literal:
                node["alt_labels"].append(str(literal))

        # Alternative labels — standard SKOS
        for alt in g.objects(s, SKOS.altLabel):
            node["alt_labels"].append(str(alt))

        # Alignment predicates
        for pred, match_type in [
            (SKOS.closeMatch,   "closeMatch"),
            (SKOS.narrowMatch,  "narrowMatch"),
            (SKOS.broadMatch,   "broadMatch"),
            (SKOS.relatedMatch, "relatedMatch"),
        ]:
            for o in g.objects(s, pred):
                if is_scopus(o):
                    node["scopus_alignments"].append({
                        "uri":  str(create_safe_uri(str(o))),
                        "type": match_type,
                    })

        results.append(node)

    return results


def parse_loc_folder(folder_path: str) -> list[dict]:
    """
    Parse all `.rdf` files in *folder_path* and return a flat list of LOC records.

    Parameters
    ----------
    folder_path : str
        Path to the directory containing LOC RDF/XML alignment files.
    """
    print(f"Parsing LOC alignment folder: {folder_path}")

    all_data = []

    rdf_files = sorted(f for f in os.listdir(folder_path) if f.endswith(".rdf"))
    print(f"  Found {len(rdf_files)} RDF file(s).")

    for filename in rdf_files:
        print(f"  Parsing: {filename}")
        all_data.extend(parse_rdf_file(os.path.join(folder_path, filename)))

    return all_data


### 5.4 Merge LOC and Scopus data

In [8]:
def merge(loc_data: list[dict], scopus_graph: dict) -> tuple[list[dict], list[str]]:
    """
    Enrich each LOC record with Scopus metadata for every alignment URI.

    For each alignment, the Scopus entry (label, area, categories, …) is looked up
    in *scopus_graph* and inlined into the LOC record.  URIs that cannot be found
    receive a ``warning`` field instead.

    Parameters
    ----------
    loc_data : list[dict]
        Output of :func:`parse_loc_folder`.
    scopus_graph : dict
        Output of :func:`parse_scopus_ttl`.

    Returns
    -------
    merged : list[dict]
        Enriched LOC records.
    missing_uris : list[str]
        URIs that were referenced in LOC files but absent from the Scopus graph.
    """
    print("Merging LOC alignments with Scopus graph...")

    merged       = []
    missing_uris = []

    for loc_entry in loc_data:
        entry = dict(loc_entry)
        enriched_alignments = []

        for alignment in loc_entry.get("scopus_alignments", []):
            uri  = alignment["uri"]
            info = scopus_graph.get(uri)

            enriched = {"uri": uri, "type": alignment["type"]}

            if info:
                enriched.update({
                    "label":           info.get("label"),
                    "area":            info.get("area", []),
                    "area_labels":     info.get("area_labels", []),
                    "categories":      info.get("categories", []),
                    "category_labels": info.get("category_labels", []),
                })
            else:
                enriched["warning"] = "URI not found in Scopus graph"
                missing_uris.append(uri)

            enriched_alignments.append(enriched)

        entry["scopus_alignments"] = enriched_alignments
        merged.append(entry)

    return merged, missing_uris


### 5.5 Run the pipeline and save output

In [9]:
# ── Run ───────────────────────────────────────────────────────────────────────
scopus_graph = parse_scopus_ttl(SCOPUS_TTL)
loc_data     = parse_loc_folder(LOC_FOLDER)

merged_data, missing_uris = merge(loc_data, scopus_graph)

# ── Save ──────────────────────────────────────────────────────────────────────
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(merged_data, f, indent=2, ensure_ascii=False)

print("\n─── Summary ───────────────────────────────────────────────")
print(f"  Merged entries : {len(merged_data)}")
print(f"  Missing URIs   : {len(set(missing_uris))}")
print(f"  Output saved to: {OUTPUT_JSON}")


Parsing Scopus TTL: data\vocabolario_SCOPUS_validated.ttl
Parsing LOC alignment folder: data\File LoC aggiornati con allineamento
  Found 159 RDF file(s).
  Parsing: AM1-AM501.skos.rdf
  Parsing: B1-B5802.skos.rdf
  Parsing: BF1-BF990.skos.rdf
  Parsing: BJ1-BJ1725.skos.rdf
  Parsing: BL1-BL2790.skos.rdf
  Parsing: CC1-CC960.skos.rdf
  Parsing: D1-D2027.skos.rdf
  Parsing: DA1-DA995.skos.rdf
  Parsing: DAW1001-DAW1051.skos.rdf
  Parsing: DB1-DB879.skos.rdf
  Parsing: DC1-DC947.skos.rdf
  Parsing: DE1-DE100.skos.rdf
  Parsing: DF10-DF951.skos.rdf
  Parsing: DG11-DG980.2.skos.rdf
  Parsing: DH1-DH207.skos.rdf
  Parsing: DJ1-DJ500.22.skos.rdf
  Parsing: DL1-DL1180.2.skos.rdf
  Parsing: DP1-DP402.skos.rdf
  Parsing: DQ1-DQ851.skos.rdf
  Parsing: DR1-DR2285.skos.rdf
  Parsing: DS1-DS937.skos.rdf
  Parsing: DT1-DT3415.skos.rdf
  Parsing: DU1-DU950.skos.rdf
  Parsing: DX101-DX301.skos.rdf
  Parsing: E11-E143.skos.rdf
  Parsing: F1-F975.skos.rdf
  Parsing: G1-G922.skos.rdf
  Parsing: GC1-GC158